In [19]:
from pathlib import Path
DATA_DIR = Path(r"D:\4-University Stuff Season 16\KI 1\Mang Noron\DuDoanHinhAnh\data")
OUT_DIR = Path(r"D:\4-University Stuff Season 16\KI 1\Mang Noron\DuDoanHinhAnh\submissions")

**CONFIG**

In [20]:
import os
import time
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import StratifiedKFold

In [21]:
DATA_DIR = Path('data')
OUT_DIR = Path('submissions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 3
EPOCHS = 80
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 2e-2
DROPOUT_RATE = 0.3
PATIENCE = 7

def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

**DATASET & DATA AUGMENTATION**

In [22]:
def load_raw_data():
    train = np.load(DATA_DIR / "train.npz")
    test = np.load(DATA_DIR / "test.npz")
    x = train["images"].transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    y = train["labels"].astype(np.int64)
    x_test = test["images"].transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    return x, y, x_test, test["ids"]


class CIFARDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels) if labels is not None else None
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, self.labels[idx]
        return img

**RESIDUAL VÀ KAIMING INIT**

In [23]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + x)


class DeepResNetModel(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, dropout_rate=0.3):
        super().__init__()
        self.in_block = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.stage1 = nn.Sequential(
            ResidualBlock(64),
            ResidualBlock(64),
            nn.MaxPool2d(2, 2)  # 32x32 -> 16x16
        )

        self.trans1 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.stage2 = nn.Sequential(
            ResidualBlock(128),
            ResidualBlock(128),
            nn.MaxPool2d(2, 2)  # 16x16 -> 8x8
        )

        self.trans2 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.stage3 = nn.Sequential(
            ResidualBlock(256),
            ResidualBlock(256),
            nn.MaxPool2d(2, 2)  # 8x8 -> 4x4
        )

        self.trans3 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.stage4 = nn.Sequential(
            ResidualBlock(512)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.in_block(x)
        x = self.stage1(x)
        x = self.trans1(x)
        x = self.stage2(x)
        x = self.trans2(x)
        x = self.stage3(x)
        x = self.trans3(x)
        x = self.stage4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

**ACCURACY & RMSE**

In [24]:
@torch.no_grad()
def evaluate(model, loader, device, num_classes=10):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    squared_error_sum = 0.0
    criterion = nn.CrossEntropyLoss(reduction="sum")

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item()

        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += len(yb)

        probs = torch.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs).scatter_(1, yb.unsqueeze(1), 1.0)
        squared_error_sum += torch.sum((probs - one_hot) ** 2).item()

    rmse = np.sqrt(squared_error_sum / (total * num_classes))
    return loss_sum / total, correct / total, rmse

**TRAING**

In [25]:
def main():
    set_seed(SEED)
    device = get_device()
    x, y, x_test, test_ids = load_raw_data()

    train_transform = T.Compose([
        T.RandomCrop(32, padding=4, padding_mode='reflect'),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.15, contrast=0.15),
    ])

    test_ds = CIFARDataset(x_test, transform=None)
    test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_val_accs = []
    oof_val_rmses = []
    test_preds_folds = []

    print(f"Bắt đầu trên {device}...\n")

    for fold, (train_idx, val_idx) in enumerate(skf.split(x, y)):
        print(f"FOLD {fold + 1}/{N_SPLITS}")

        train_ds = CIFARDataset(x[train_idx], y[train_idx], transform=train_transform)
        val_ds = CIFARDataset(x[val_idx], y[val_idx], transform=None)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

        model = DeepResNetModel(dropout_rate=DROPOUT_RATE).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        warmup_epochs = 3
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
        cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(EPOCHS - warmup_epochs), eta_min=1e-6)
        scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

        best_val_acc = 0.0
        best_val_rmse = 0.0
        best_model_weights = None
        no_improve_epochs = 0

        #scaler = torch.cuda.amp.GradScaler() #tối ưu cho rtx 4060

        for epoch in range(1, EPOCHS + 1):
            model.train()
            t0 = time.time()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()

            scheduler.step()

            tr_loss, tr_acc, _ = evaluate(model, train_loader, device)
            va_loss, va_acc, va_rmse = evaluate(model, val_loader, device)

            print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
                  f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} RMSE: {va_rmse:.4f} | {time.time() - t0:.1f}s")

            if va_acc > best_val_acc:
                best_val_acc = va_acc
                best_val_rmse = va_rmse
                best_model_weights = copy.deepcopy(model.state_dict())
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1

            if no_improve_epochs >= PATIENCE:
                print(f"Early Stopping tại Epoch {epoch}.")
                break

        print(f"\n=> Kết quả tốt nhất Fold {fold + 1}: Val Acc = {best_val_acc:.4f} | Val RMSE = {best_val_rmse:.4f}\n")
        oof_val_accs.append(best_val_acc)
        oof_val_rmses.append(best_val_rmse)

        model.load_state_dict(best_model_weights)
        model.eval()
        fold_probs = []
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(device)
                probs = torch.softmax(model(xb), dim=1).cpu().numpy()
                fold_probs.append(probs)
        test_preds_folds.append(np.concatenate(fold_probs, axis=0))

    print("TỔNG KẾT")
    print(f"Mean Val Acc: {np.mean(oof_val_accs):.4f} +/- {np.std(oof_val_accs):.4f}")
    print(f"Mean Val RMSE: {np.mean(oof_val_rmses):.4f} +/- {np.std(oof_val_rmses):.4f}")

    avg_test_probs = np.mean(test_preds_folds, axis=0)
    final_preds = np.argmax(avg_test_probs, axis=1)

    out_file = OUT_DIR / "submission_5.csv"
    with open(out_file, "w") as f:
        f.write("id,label\n")
        for i, p in zip(test_ids, final_preds):
            f.write(f"{i},{p}\n")
    print(f"\nĐã ghi file tại: {out_file}")


if __name__ == "__main__":
    main()

Bắt đầu trên cuda...

FOLD 1/3
Epoch  1/80 | Train Loss: 2.7075 Acc: 0.3224 | Val Loss: 2.4688 Acc: 0.3496 RMSE: 0.2983 | 47.5s
Epoch  2/80 | Train Loss: 3.2548 Acc: 0.3528 | Val Loss: 2.5779 Acc: 0.3917 RMSE: 0.2800 | 45.1s


d:\4-University Stuff Season 16\KI 1\Mang Noron\DuDoanHinhAnh\ai_env\lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch  3/80 | Train Loss: 3.6856 Acc: 0.3701 | Val Loss: 3.5125 Acc: 0.3836 RMSE: 0.2824 | 45.5s
Epoch  4/80 | Train Loss: 1.5070 Acc: 0.4772 | Val Loss: 1.3913 Acc: 0.5206 RMSE: 0.2479 | 47.3s
Epoch  5/80 | Train Loss: 1.5894 Acc: 0.4893 | Val Loss: 1.4654 Acc: 0.5257 RMSE: 0.2469 | 44.3s
Epoch  6/80 | Train Loss: 1.1418 Acc: 0.5968 | Val Loss: 1.0208 Acc: 0.6373 RMSE: 0.2199 | 44.4s
Epoch  7/80 | Train Loss: 1.0130 Acc: 0.6405 | Val Loss: 0.8962 Acc: 0.6849 RMSE: 0.2074 | 44.5s
Epoch  8/80 | Train Loss: 1.3677 Acc: 0.6219 | Val Loss: 1.1695 Acc: 0.6593 RMSE: 0.2155 | 44.4s
Epoch  9/80 | Train Loss: 0.9361 Acc: 0.6754 | Val Loss: 0.8125 Acc: 0.7199 RMSE: 0.1960 | 44.4s
Epoch 10/80 | Train Loss: 1.7956 Acc: 0.4975 | Val Loss: 1.4650 Acc: 0.5807 RMSE: 0.2426 | 45.8s
Epoch 11/80 | Train Loss: 1.1940 Acc: 0.6215 | Val Loss: 0.9980 Acc: 0.6808 RMSE: 0.2114 | 45.2s
Epoch 12/80 | Train Loss: 0.8910 Acc: 0.7175 | Val Loss: 0.7850 Acc: 0.7596 RMSE: 0.1832 | 44.4s
Epoch 13/80 | Train Loss: 0.76